In [0]:
#Notebook para estudo e tesde de usabilidade do Genie Code. Genie Code possui skills próprias voltadas para o uso do Databricks
#pedi pra ele validações em contextos de valores e negócio. Ele vasculhou principalmente os metadados que preparamos na silver.
#Com outros agentes, temos que ter a pastinha de skills e contexto pra ele trabalhar
#Se não conhecer de algum tópico que precisa usar, pede pra IA gerar e você vai aprendendo junto, ao invés de correr atrás e demorar muito tempo
#vamos fazer uma avaliação na tabela da silver, em aspecto de qualidade, ver o que podemos tratar

from pyspark.sql.functions import col, count, when

#carregar tabela
df = spark.table("silver.aerodromos")

#verificação de valores nulos
display(df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]))

#verificação de duplicidade pela chave primária provável (icao)
display(df.groupBy("icao").count().filter("count > 1"))

#Verificação de distribuição de valores em colunas de categorias (uf_nome, situacao)
display(df.groupBy("uf").count().orderBy("count"))
display(df.groupBy("situacao").count())

In [0]:
# === VALIDAÇÕES DE VALORES: formato, domínio, unicidade, range ===
from pyspark.sql.functions import col, count, when, trim, min as spark_min, max as spark_max

df = spark.table("voebem.silver.aerodromos")

# 1. ICAO — formato (regex corrigida: permite dígitos nas posições 3-4)
print("=== 1. ICAO — FORMATO ===")
print("Regra: ^[A-Z]{2}[A-Z0-9]{2}$ (4 chars, 2 letras + 2 alfanuméricos)\n")
invalid_icao = df.filter(~col("icao").rlike("^[A-Z]{2}[A-Z0-9]{2}$"))
print(f"Fora do formato: {invalid_icao.count()}")
if invalid_icao.count() > 0:
    display(invalid_icao.select("icao", "ciad", "nome", "uf_nome"))

# 2. ICAO — prefixo brasileiro (SB, SW, SN, SD, SS, SI)
print("=== 2. ICAO — PREFIXO BRASILEIRO ===")
print("Esperados: SB, SW, SN, SD, SS, SI\n")
display(df.withColumn("prefixo", col("icao").substr(1, 2))
    .groupBy("prefixo").agg(count("*").alias("total")).orderBy(col("total").desc()))

# 3. CIAD — formato + unicidade
print("=== 3. CIAD — FORMATO E UNICIDADE ===")
print("Regra: ^[A-Z]{2}\\d{4}$ (2 letras + 4 dígitos), único\n")
invalid_ciad = df.filter(~col("ciad").rlike("^[A-Z]{2}\\d{4}$"))
dup_ciad = df.groupBy("ciad").agg(count("*").alias("count")).filter(col("count") > 1)
print(f"Fora do formato: {invalid_ciad.count()} | Duplicados: {dup_ciad.count()}")
if dup_ciad.count() > 0:
    display(dup_ciad)

# 4. Altitude — range + estatísticas
print("=== 4. ALTITUDE — RANGE E ESTATÍSTICAS ===")
print("Regra: 0 a 3000m (faixa plausível Brasil)\n")
display(df.select("altitude_m").summary("count", "min", "25%", "50%", "75%", "max"))
alt_out = df.filter((col("altitude_m") < 0) | (col("altitude_m") > 3000))
alt_zero = df.filter(col("altitude_m") == 0)
print(f"Fora do range: {alt_out.count()} | Altitude = 0 (nível do mar): {alt_zero.count()}")
if alt_zero.count() > 0:
    display(alt_zero.select("icao", "nome", "municipio", "altitude_m"))

# 5. Situação — domínio esperado
print("=== 5. SITUAÇÃO — DOMÍNIO ===")
print("Esperados: Cadastrado, Interditado, Desativado, Construção\n")
display(df.groupBy("situacao").agg(count("*").alias("total")))
unexpected = df.filter(~col("situacao").isin("Cadastrado", "Interditado", "Desativado", "Construção") & col("situacao").isNotNull())
print(f"Valores fora do domínio: {unexpected.count()}")

# 6. Cobertura de UFs
print("=== 6. COBERTURA DE UFs ===")
print("Esperado: 27 UFs (26 estados + DF)\n")
display(df.groupBy("uf_nome").agg(count("*").alias("total")).orderBy(col("total").desc()))
n_ufs = df.filter(col("uf_nome").isNotNull()).select("uf_nome").distinct().count()
print(f"UFs distintas: {n_ufs} | UF nula: {df.filter(col('uf_nome').isNull()).count()}")

# 7. Timestamps de auditoria — _transformado_em >= _ingerido_em
print("=== 7. TIMESTAMPS DE AUDITORIA ===")
print("Regra: _transformado_em deve ser >= _ingerido_em\n")
ts_bad = df.filter(col("_transformado_em") < col("_ingerido_em"))
print(f"Registros inconsistentes: {ts_bad.count()}")

# 8. Nome — não vazio
print("=== 8. NOME — NÃO VAZIO ===")
empty_nome = df.filter(col("nome").isNull() | (trim(col("nome")) == ""))
print(f"Registros com nome vazio: {empty_nome.count()}")

In [0]:
# === VALIDAÇÕES DE NEGÓCIO: consistência, cross-field, cross-table ===
from pyspark.sql.functions import col, count, when, collect_set, size, trim

df = spark.table("voebem.silver.aerodromos")

# 1. CIAD vs UF — sigla do CIAD deve corresponder à UF
print("=== 1. CIAD vs UF — CONSISTÊNCIA DE SIGLA ===")
print("Regra: 2 primeiros chars do CIAD = sigla da UF\n")

sigla_map = (
    when(col("uf_nome") == "Acre", "AC")
    .when(col("uf_nome") == "Alagoas", "AL")
    .when(col("uf_nome") == "Amapá", "AP")
    .when(col("uf_nome") == "Amazonas", "AM")
    .when(col("uf_nome") == "Bahia", "BA")
    .when(col("uf_nome") == "Ceará", "CE")
    .when(col("uf_nome") == "Distrito Federal", "DF")
    .when(col("uf_nome") == "Espírito Santo", "ES")
    .when(col("uf_nome") == "Goiás", "GO")
    .when(col("uf_nome") == "Maranhão", "MA")
    .when(col("uf_nome") == "Mato Grosso", "MT")
    .when(col("uf_nome") == "Mato Grosso do Sul", "MS")
    .when(col("uf_nome") == "Minas Gerais", "MG")
    .when(col("uf_nome") == "Pará", "PA")
    .when(col("uf_nome") == "Paraíba", "PB")
    .when(col("uf_nome") == "Paraná", "PR")
    .when(col("uf_nome") == "Pernambuco", "PE")
    .when(col("uf_nome") == "Piauí", "PI")
    .when(col("uf_nome") == "Rio de Janeiro", "RJ")
    .when(col("uf_nome") == "Rio Grande do Norte", "RN")
    .when(col("uf_nome") == "Rio Grande do Sul", "RS")
    .when(col("uf_nome") == "Rondônia", "RO")
    .when(col("uf_nome") == "Roraima", "RR")
    .when(col("uf_nome") == "Santa Catarina", "SC")
    .when(col("uf_nome") == "São Paulo", "SP")
    .when(col("uf_nome") == "Sergipe", "SE")
    .when(col("uf_nome") == "Tocantins", "TO")
    .otherwise(None)
)

df_chk = df.withColumn("sigla_uf", sigla_map).withColumn("sigla_ciad", col("ciad").substr(1, 2))
mismatch = df_chk.filter(col("sigla_uf").isNotNull() & (col("sigla_ciad") != col("sigla_uf")))
print(f"CIAD-UF inconsistente: {mismatch.count()}")
if mismatch.count() > 0:
    display(mismatch.select("icao", "ciad", "nome", "uf_nome", "sigla_uf", "sigla_ciad"))

# 2. Municipio vs municipio_servido — casos cross-UF
print("=== 2. MUNICIPIO vs MUNICIPIO_SERVIDO — CROSS-UF ===")
print("Regra: aeródromo e município servido geralmente na mesma UF\n")
cross_uf = df.filter(
    col("uf_nome").isNotNull() & col("uf_servido_nome").isNotNull() &
    (col("uf_nome") != col("uf_servido_nome")))
print(f"Aeródromos onde UF ≠ UF servida: {cross_uf.count()}")
if cross_uf.count() > 0:
    display(cross_uf.select("icao", "nome", "municipio", "uf_nome", "municipio_servido", "uf_servido_nome"))

# 3. Interditados por UF — concentração anômala
print("=== 3. AERÓDROMOS INTERDITADOS POR UF ===")
print("Regra: verificar distribuição de interditados para identificar concentrações\n")
display(df.filter(col("situacao") == "Interditado")
    .groupBy("uf_nome").agg(count("*").alias("interditados"))
    .orderBy(col("interditados").desc()))

# 4. Nomes duplicados (mesmo nome, ICAO diferente)
print("=== 4. NOMES DUPLICADOS ===")
print("Regra: mesmo nome com ICAO diferente pode indicar erro de cadastro ou coincidência\n")
dup_nomes = (df.filter(col("nome").isNotNull())
    .groupBy("nome").agg(count("*").alias("count"), collect_set("icao").alias("icaos"))
    .filter(col("count") > 1))
print(f"Nomes duplicados: {dup_nomes.count()}")
if dup_nomes.count() > 0:
    display(dup_nomes.withColumn("n_icaos", size("icaos")))

# 5. Referência cruzada com VRA — cobertura de operações
print("=== 5. REFERÊNCIA CRUZADA COM VRA ===")
print("Verifica: aeródromos da silver vs ICAOs usados no VRA\n")

vra = spark.table("voebem.silver.vra")
vra_icaos = (vra.select(col("icao_origem").alias("icao"))
    .union(vra.select(col("icao_destino").alias("icao")))
    .distinct())

in_vra = df.join(vra_icaos, "icao", "inner")
print(f"Aeródromos usados no VRA: {in_vra.count()} / {df.count()} total")

orphaned = df.join(vra_icaos, "icao", "left_anti")
print(f"Aeródromos brasileiros SEM operação no VRA: {orphaned.count()}")
if orphaned.count() > 0:
    display(orphaned.select("icao", "nome", "uf_nome", "situacao").orderBy("uf_nome", "nome"))

foreign = vra_icaos.join(df, "icao", "left_anti")
print(f"ICAOs no VRA fora da silver (prováveis estrangeiros): {foreign.count()}")